# 道路封闭或拥堵后，怎样修订剩余计划？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

{
  "strategies": [
    {
      "policy": "fixed",
      "success": false,
      "time": 1,
      "changes": 0,
      "reason": "OLD_PLAN_INFEASIBLE"
    },
    {
      "policy": "repair_blocked",
      "success": true,
      "time": 12,
      "changes": 1
    },
    {
      "policy": "rolling",
      "success": true,
      "time": 5,
      "changes": 2
    }
  ]
}
本情景推荐rolling，实际耗时5、修改2次。仅在封路时修复会忽视仍可通行但变慢的道路；可将成本明显变化也设为事件触发。本例是图上反馈重规划，没有声称具备连续控制MPC的稳定性保证。


## 1. 补全：辨认不可行的旧计划
回放时刻1事件，检查原路线哪条边已经失效。

In [2]:
g=initial.copy();seen=set();observe(g,1,seen)
invalid=[(u,v) for u,v in zip(old_path,old_path[1:]) if not g.has_edge(u,v)]
assert invalid==[("A","G")]
print(invalid)


[('A', 'G')]


## 2. 修改：没有任何环境事件
暂时设事件为空，重新比较三种策略。完成后恢复事件，避免后续单元格使用错误环境。

In [3]:
saved_events=events
try:
    events=[]
    unchanged=[execute(p) for p in ["fixed","repair_blocked","rolling"]]
    assert all(r["success"] and r["time"]==2 for r in unchanged)
    assert all(r["changes"]==0 for r in unchanged)
finally: events=saved_events
print("稳定环境下，本例反复重算没有路线收益。")


稳定环境下，本例反复重算没有路线收益。


## 3. 迁移：物流执行审计
复制滚动策略轨迹，篡改一个道路耗时，验证审计器能发现不一致。

In [4]:
import copy
corrupted=copy.deepcopy(results[2])
corrupted["trace"][0]["duration"]+=3
assert not verify_execution(corrupted)
assert verify_execution(results[2])
print("执行日志必须与当时可观测数据一致。")


执行日志必须与当时可观测数据一致。
